https://apisidra.ibge.gov.br/values/t/1612/n1/1/v/allxp/p/last 35/c81/allxt/h/n?formato=json


Exemplo de resposta: 





https://apisidra.ibge.gov.br/values/t/1613/n1/1/p/last 35/v/all/c82/allxt/f/a/h/n?formato=json


d4c e d4n

In [2]:
import requests
import pandas as pd
import random
import time

anos = range(1, 15)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

url_1612 = "https://apisidra.ibge.gov.br/values/t/1612/n3/all/v/216/p/last{ano}/c81/all?formato=json"
url_1613 = "https://apisidra.ibge.gov.br/values/t/1613/n3/all/v/216/p/last{ano}/c82/allxt/f/a/h/n?formato=json"

def _limpar_linha_cabecalho(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    row0 = df.iloc[0].astype(str)
    if row0.str.contains(r"Nível|Unidade|\(Código\)|\(\%\)", case=False, regex=True).any():
        return df.iloc[1:].reset_index(drop=True)
    return df

def _aplicar_ano_na_url(url: str, ano: int) -> str:
    if "{ano}" in url:
        return url.replace("{ano}", str(ano))
    if "{last_ano}" in url:
        return url.replace("{last_ano}", str(ano))
    return url

def carregar_sidra(url: str, anos_iter=anos) -> pd.DataFrame:
    dados_combinados = []

    if "{ano}" not in url and "{last_ano}" not in url:
        try:
            r = requests.get(url, timeout=30, headers=HEADERS)
            print(f"Status: {r.status_code}, Response length: {len(r.text)}")
            r.raise_for_status()
            if not r.text or "Request Rejected" in r.text:
                print(f"Erro na resposta para: {url}")
                return pd.DataFrame()
            return _limpar_linha_cabecalho(pd.DataFrame(r.json()))
        except Exception as e:
            print(f"Erro ao carregar URL única: {e}")
            return pd.DataFrame()

    teve_sucesso = False
    for ano in anos_iter:
        url_ano = _aplicar_ano_na_url(url, ano)
        try:
            r = requests.get(url_ano, timeout=30, headers=HEADERS)
            if r.status_code == 200 and r.text and "Request Rejected" not in r.text:
                bloco = r.json()
                if isinstance(bloco, list) and bloco:
                    if not dados_combinados:
                        dados_combinados.extend(bloco)
                    else:
                        dados_combinados.extend(bloco[1:])
                    teve_sucesso = True
            else:
                print(f"Ano {ano}: status {r.status_code} | URL: {url_ano}")
        except Exception as e:
            print(f"Ano {ano}: erro {e}")

        time.sleep(0.35 + random.random() * 0.25)

    if not teve_sucesso:
        return pd.DataFrame()

    df = pd.DataFrame(dados_combinados)
    return _limpar_linha_cabecalho(df)

def preparar_tabela_1612(df_1612: pd.DataFrame) -> pd.DataFrame:
    cols_1612 = [c for c in ["D1C", "D1N", "D3N", "D4C", "D4N", "D2N", "MN", "V"] if c in df_1612.columns]
    t1612 = df_1612[cols_1612].copy() if cols_1612 else pd.DataFrame()
    if t1612.empty:
        return t1612

    t1612 = t1612.rename(
        columns={
            "D1C": "geocodigo",
            "D1N": "nome_uf",
            "D3N": "ano",
            "D4C": "codigo",
            "D4N": "cultura",
            "D2N": "producao",
            "MN": "unidade",
            "V": "valor",
        }
    )
    t1612["valor"] = pd.to_numeric(t1612["valor"], errors="coerce")
    t1612["tipo"] = "temporario"
    return t1612

def preparar_tabela_1613(df_1613: pd.DataFrame) -> pd.DataFrame:
    cols_1613 = [c for c in ["D1C", "D1N", "D2N", "D4C", "D4N", "D3N", "MN", "V"] if c in df_1613.columns]
    t1613 = df_1613[cols_1613].copy() if cols_1613 else pd.DataFrame()
    if t1613.empty:
        return t1613

    t1613 = t1613.rename(
        columns={
            "D1C": "geocodigo",
            "D1N": "nome_uf",
            "D4C": "codigo",
            "D4N": "cultura",
            "D3N": "ano",
            "D2N": "producao",
            "MN": "unidade",
            "V": "valor",
        }
    )
    t1613["valor"] = pd.to_numeric(t1613["valor"], errors="coerce")
    t1613["tipo"] = "permanente"
    return t1613

df_1612 = carregar_sidra(url_1612)
df_1613 = carregar_sidra(url_1613)

t1612 = preparar_tabela_1612(df_1612)
t1613 = preparar_tabela_1613(df_1613)

In [20]:
t1612

,geocodigo,nome_uf,ano,codigo,cultura,producao,unidade,valor,tipo
0,11,Rondônia,2024,0,Total,Área colhida,Hectares,1057373.0,temporario
1,11,Rondônia,2024,2688,Abacaxi*,Área colhida,Hectares,1095.0,temporario
2,11,Rondônia,2024,40471,Alfafa fenada,Área colhida,Hectares,NaN,temporario
3,11,Rondônia,2024,2689,Algodão herbáceo (em caroço),Área colhida,Hectares,9862.0,temporario
4,11,Rondônia,2024,2690,Alho,Área colhida,Hectares,NaN,temporario
...,...,...,...,...,...,...,...,...,...
96385,53,Distrito Federal,2024,2713,Soja (em grão),Área colhida,Hectares,85000.0,temporario
96386,53,Distrito Federal,2024,2714,Sorgo (em grão),Área colhida,Hectares,22000.0,temporario
96387,53,Distrito Federal,2024,2715,Tomate,Área colhida,Hectares,350.0,temporario
96388,53,Distrito Federal,2024,2716,Trigo (em grão),Área colhida,Hectares,5300.0,temporario


In [ ]:
t1613.head()

In [ ]:
tabela_final = pd.concat([t1612, t1613], ignore_index=True, sort=False)

if not tabela_final.empty:
    if "ano" in tabela_final.columns:
        tabela_final["ano_num"] = pd.to_numeric(tabela_final["ano"], errors="coerce")
    tabela_final = tabela_final.sort_values(
        [c for c in ["tipo", "cultura", "ano_num"] if c in tabela_final.columns],
        na_position="last"
    ).drop(columns=["ano_num"], errors="ignore").reset_index(drop=True)

tabela_final

In [70]:
def pivotar_valores_por_ano(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    base = df.copy()
    colunas_chave = [
        "geocodigo",
        "nome_uf",
        "codigo",
        "cultura",
        "producao",
        "unidade",
    ]

    faltando = [c for c in colunas_chave + ["ano", "valor"] if c not in base.columns]
    if faltando:
        raise ValueError(f"Colunas obrigatorias ausentes: {faltando}")

    base["valor"] = pd.to_numeric(base["valor"], errors="coerce").fillna(0)
    base["ano"] = pd.to_numeric(base["ano"], errors="coerce")
    base = base.dropna(subset=["ano"])
    base["ano"] = base["ano"].astype(int).astype(str)

    tabela = base.pivot_table(
        index=colunas_chave,
        columns="ano",
        values="valor",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()

    colunas_ano = sorted([c for c in tabela.columns if c not in colunas_chave], key=int)
    return tabela[colunas_chave + colunas_ano]

tabela_pivot_1612 = pivotar_valores_por_ano(t1612)
tabela_pivot_1613 = pivotar_valores_por_ano(t1613)

Baixar os dados da Tabela 1618 do IBGE, referente a safra de 2025. 

In [14]:
import re

url_1618 = "https://apisidra.ibge.gov.br/values/t/1618/n3/all/v/216/p/all/c49/all/c48/all?formato=json"

# Mapeamento de colunas do SIDRA → nomes legíveis
_MAP_1618 = {
    "MN":  "unidade",
    "V":   "valor",
    "D1C": "geocodigo",
    "D1N": "nome_uf",
    "D2N": "producao",          # ex: "Área colhida"
    # "D4N": "codigo_safra",      # ex: 3009 (Código Safra)
    "D3C": "mes_codigo",        # ex: "202602"  (YYYYMM)
    "D3N": "mes",               # ex: "fevereiro 2026"
    "D4C": "codigo_safra",    # ex: "3009" (Código da cultura)
    "D4N": "safra",             # ex: "Safra 2025"
    "D5C": "safra_codigo",      # ex: "39428"
    "D5N": "ano_safra"          # ex: "1.2 Amendoim (1ª Safra)"
}


In [15]:
df_1618 = carregar_sidra(url_1618)
# t1618 = preparar_tabela_1618(df_1618)
df_1618.head()

Status: 200, Response length: 487636


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N,D4C,D4N,D5C,D5N
0,3,Unidade da Federação,1006,Hectares,1340707,11,Rondônia,216,Área colhida,202603,março 2026,3009,Safra 2025,0,Total
1,3,Unidade da Federação,1006,Hectares,1262995,11,Rondônia,216,Área colhida,202603,março 2026,3009,Safra 2025,39428,"1 Cereais, leguminosas e oleaginosas"
2,3,Unidade da Federação,1006,Hectares,8915,11,Rondônia,216,Área colhida,202603,março 2026,3009,Safra 2025,39429,1.1 Algodão herbáceo
3,3,Unidade da Federação,1006,Hectares,-,11,Rondônia,216,Área colhida,202603,março 2026,3009,Safra 2025,39430,1.2 Amendoim (1ª Safra)
4,3,Unidade da Federação,1006,Hectares,-,11,Rondônia,216,Área colhida,202603,março 2026,3009,Safra 2025,39431,1.3 Amendoim (2ª Safra)


In [16]:
# Debug: inspecionar dados de 1618
colunas = [c for c in _MAP_1618 if c in df_1618.columns]
t = df_1618[colunas].copy()
t = t.rename(columns={k: v for k, v in _MAP_1618.items() if k in colunas})
t["valor"] = pd.to_numeric(t["valor"], errors="coerce")
if "mes_codigo" in t.columns:
    t["ano"] = t["mes_codigo"].astype(str).str[:4]
t["safra"] = t["ano_safra"].str.extract(r'\((\d+ª Safra)\)', flags=re.IGNORECASE)
com_safra = t[t["safra"].notna()].copy()

print("Colunas em com_safra:")
print(com_safra.columns.tolist())
print("\nTipo de dados:")
print(com_safra.dtypes)
print("\nPrimeiros valores de safra:")
print(com_safra["safra"].head(20))
print("\nValores únicos de safra:")
print(com_safra["safra"].unique())
print(f"\nTotal de linhas: {len(com_safra)}")
print(f"\nInfo sobre safra:")
print(com_safra["safra"].value_counts())

Colunas em com_safra:
['unidade', 'valor', 'geocodigo', 'nome_uf', 'producao', 'mes_codigo', 'mes', 'codigo_safra', 'safra', 'safra_codigo', 'ano_safra', 'ano']

Tipo de dados:
unidade             str
valor           float64
geocodigo           str
nome_uf             str
producao            str
mes_codigo          str
mes                 str
codigo_safra        str
safra               str
safra_codigo        str
ano_safra           str
ano                 str
dtype: object

Primeiros valores de safra:
3     1ª Safra
4     2ª Safra
10    1ª Safra
11    2ª Safra
12    3ª Safra
16    1ª Safra
17    2ª Safra
23    1ª Safra
24    2ª Safra
25    3ª Safra
40    1ª Safra
41    2ª Safra
47    1ª Safra
48    2ª Safra
49    3ª Safra
53    1ª Safra
54    2ª Safra
60    1ª Safra
61    2ª Safra
62    3ª Safra
Name: safra, dtype: str

Valores únicos de safra:
<StringArray>
['1ª Safra', '2ª Safra', '3ª Safra']
Length: 3, dtype: str

Total de linhas: 540

Info sobre safra:
safra
1ª Safra    216
2ª Saf

In [17]:
# Debug: testar pivot
com_safra["cultura"] = (
    com_safra["ano_safra"]
    .str.replace(r'\s*\(\d+ª Safra\)', '', regex=True)
    .str.strip()
)

colunas_chave = [
    c for c in ["geocodigo", "nome_uf", "codigo_safra", "cultura", 
                    "ano", "producao", "mes", "unidade", "safra", 
                    "codigo_safra", "ano_safra", "safra_codigo"]
    if c in com_safra.columns
]

print("Colunas-chave para indexar:")
print(colunas_chave)

# Verificar duplicatas no índice
teste_pivot = com_safra[colunas_chave + ["safra", "valor"]]
print("\nPrimeiro teste - shape antes do pivot:")
print(teste_pivot.shape)

# Verificar combinações únicas
print("\nCombinações únicas de índice:")
print(teste_pivot[colunas_chave].drop_duplicates().shape)

print("\nTestando pivot_table...")
try:
    result = teste_pivot.pivot_table(
        index=colunas_chave,
        columns="safra",
        values="valor",
        aggfunc="sum",
        fill_value=0,
    )
    print("✓ Pivot bem-sucedido!")
    print(result.head())
except Exception as e:
    print(f"✗ Erro: {e}")
    print(f"Tipo do erro: {type(e).__name__}")

Colunas-chave para indexar:
['geocodigo', 'nome_uf', 'codigo_safra', 'cultura', 'ano', 'producao', 'mes', 'unidade', 'safra', 'codigo_safra', 'ano_safra', 'safra_codigo']

Primeiro teste - shape antes do pivot:
(540, 14)

Combinações únicas de índice:
(540, 15)

Testando pivot_table...
✗ Erro: Grouper for 'codigo_safra' not 1-dimensional
Tipo do erro: ValueError


In [60]:
def preparar_tabela_1618(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    colunas = [c for c in _MAP_1618 if c in df.columns]
    t = df[colunas].copy()
    t = t.rename(columns={k: v for k, v in _MAP_1618.items() if k in colunas})
    t = t.loc[:, ~t.columns.duplicated()].copy()

    if "valor" not in t.columns or "ano_safra" not in t.columns:
        return pd.DataFrame()

    t["valor"] = pd.to_numeric(t["valor"], errors="coerce")

    base = t.copy()
    base["safra_etapa"] = base["ano_safra"].str.extract(
        r'\((\d+ª Safra)\)', expand=False, flags=re.IGNORECASE
    )
    base = base[base["safra_etapa"].notna()].copy()

    if base.empty:
        return pd.DataFrame()

    base["cultura"] = (
        base["ano_safra"]
        .str.replace(r'\s*\(\d+ª Safra\)', '', regex=True)
        .str.replace(r'^\d+(?:\.\d+)*\s+', '', regex=True)
        .str.strip()
    )

    index_cols = [
        c for c in ["geocodigo", "nome_uf", "producao", "cultura", "mes", "safra", "unidade"]
        if c in base.columns
    ]
    if "safra" in base.columns:
        base["safra"] = (
            base["safra"]
            .astype(str)
            .str.extract(r"(\d{4})", expand=False)
        )
        base = base[base["safra"].notna()].copy()
    if not index_cols:
        return pd.DataFrame()

    base_clean = base.dropna(subset=["safra_etapa", "valor"])

    tabela_pivot = (
        base_clean.groupby(index_cols + ["safra_etapa"], observed=False)["valor"]
        .sum()
        .unstack("safra_etapa", fill_value=0)
        .reset_index()
    )
    tabela_pivot.columns.name = None

    safras = sorted(
        [c for c in tabela_pivot.columns if c not in index_cols],
        key=lambda s: int(re.search(r"(\d+)ª", str(s)).group(1)) if re.search(r"(\d+)ª", str(s)) else 999,
    )

    return tabela_pivot[index_cols + safras]


In [61]:
t1618 = preparar_tabela_1618(df_1618)
t1618

,geocodigo,nome_uf,producao,cultura,mes,safra,unidade,1ª Safra,2ª Safra,3ª Safra
0,11,Rondônia,Área colhida,Feijão,março 2026,2025,Hectares,2380.0,110.0,0.0
1,11,Rondônia,Área colhida,Feijão,março 2026,2026,Hectares,1531.0,331.0,0.0
2,11,Rondônia,Área colhida,Milho,março 2026,2025,Hectares,14027.0,476190.0,0.0
3,11,Rondônia,Área colhida,Milho,março 2026,2026,Hectares,13915.0,520273.0,0.0
4,12,Acre,Área colhida,Amendoim,março 2026,2025,Hectares,20.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
159,53,Distrito Federal,Área colhida,Batata - inglesa,março 2026,2026,Hectares,0.0,100.0,0.0
160,53,Distrito Federal,Área colhida,Feijão,março 2026,2025,Hectares,8000.0,100.0,8000.0
161,53,Distrito Federal,Área colhida,Feijão,março 2026,2026,Hectares,10000.0,100.0,8000.0
162,53,Distrito Federal,Área colhida,Milho,março 2026,2025,Hectares,15000.0,44300.0,0.0


In [62]:
# Reorganizar t1618 com hierarquia: safra (Safra 2025/2026) como categoria superior
# e 1ª Safra, 2ª Safra, 3ª Safra como sub-classificações

colunas_index = [c for c in ["geocodigo", "nome_uf", "producao", "cultura", "unidade"] if c in t1618.columns]
colunas_safra = [c for c in t1618.columns if "ª Safra" in str(c)]
safras_unicas = sorted(t1618["safra"].unique()) if "safra" in t1618.columns else []

# Criar MultiIndex de colunas: (Safra 2025 > 1ª Safra), (Safra 2025 > 2ª Safra), etc.
frames = []
for safra_cat in safras_unicas:
    sub = t1618[t1618["safra"] == safra_cat][colunas_index + colunas_safra].copy()
    sub = sub.set_index(colunas_index)
    sub.columns = pd.MultiIndex.from_tuples(
        [(safra_cat, col) for col in sub.columns],
        names=["Safra", "Etapa"]
    )
    frames.append(sub)

t1618_hierarquico = pd.concat(frames, axis=1).reset_index()

# Exibir com MultiIndex de colunas
print("Shape:", t1618_hierarquico.shape)
print("\nColunas:")
print(t1618_hierarquico.columns.tolist())
t1618_hierarquico

Shape: (82, 11)

Colunas:
[('geocodigo', ''), ('nome_uf', ''), ('producao', ''), ('cultura', ''), ('unidade', ''), ('2025', '1ª Safra'), ('2025', '2ª Safra'), ('2025', '3ª Safra'), ('2026', '1ª Safra'), ('2026', '2ª Safra'), ('2026', '3ª Safra')]


Safra geocodigo           nome_uf      producao           cultura   unidade  \
Etapa                                                                         
0            11          Rondônia  Área colhida            Feijão  Hectares   
1            11          Rondônia  Área colhida             Milho  Hectares   
2            12              Acre  Área colhida          Amendoim  Hectares   
3            12              Acre  Área colhida            Feijão  Hectares   
4            12              Acre  Área colhida             Milho  Hectares   
..          ...               ...           ...               ...       ...   
77           52             Goiás  Área colhida            Feijão  Hectares   
78           52             Goiás  Área colhida             Milho  Hectares   
79           53  Distrito Federal  Área colhida  Batata - inglesa  Hectares   
80           53  Distrito Federal  Área colhida            Feijão  Hectares   
81           53  Distrito Federal  Área colhida             Milho  Hectares   

Safra      2025                          2026                      
Etapa  1ª Safra   2ª Safra 3ª Safra  1ª Safra   2ª Safra 3ª Safra  
0        2380.0      110.0      0.0    1531.0      331.0      0.0  
1       14027.0   476190.0      0.0   13915.0   520273.0      0.0  
2          20.0        0.0      0.0      10.0        0.0      0.0  
3           0.0     5033.0      0.0       0.0     4998.0      0.0  
4       28510.0    10578.0      0.0   30667.0    11525.0      0.0  
..          ...        ...      ...       ...        ...      ...  
77      38345.0    12670.0  90723.0   35196.0    12340.0  88799.0  
78     161994.0  2208142.0      0.0  169406.0  2155129.0      0.0  
79          0.0      100.0      0.0       0.0      100.0      0.0  
80       8000.0      100.0   8000.0   10000.0      100.0   8000.0  
81      15000.0    44300.0      0.0   14485.0    40000.0      0.0  

[82 rows x 11 columns]

In [7]:
import openpyxl

In [45]:
import importlib.util
from pathlib import Path

def salvar_dataframe_excel(df, nome_arquivo="tabela.xlsx", pasta=None, index=False):
    if df is None or df.empty:
        raise ValueError("DataFrame vazio ou inexistente.")

    pasta = Path(pasta) if pasta else Path.cwd()
    pasta.mkdir(parents=True, exist_ok=True)
    caminho = pasta / nome_arquivo

    if importlib.util.find_spec("openpyxl"):
        engine = "openpyxl"
        df.to_excel(caminho, index=index, engine=engine)
        print(f"Excel salvo em: {caminho.resolve()} (engine={engine})")
        return caminho

    if importlib.util.find_spec("xlsxwriter"):
        engine = "xlsxwriter"
        df.to_excel(caminho, index=index, engine=engine)
        print(f"Excel salvo em: {caminho.resolve()} (engine={engine})")
        return caminho

    fallback = caminho.with_suffix(".csv")
    df.to_csv(fallback, index=index)
    print(f"openpyxl/xlsxwriter não instalados. CSV salvo em: {fallback.resolve()}")
    return fallback


In [63]:
t1618 = t1618.rename(columns={"safra": "ano"})
salvar_dataframe_excel(t1618, "t1618_safras_areas.xlsx")

Excel salvo em: /home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/t1618_safras_areas.xlsx (engine=openpyxl)


PosixPath('/home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/t1618_safras_areas.xlsx')

from bs4 import BeautifulSoup

# Monta tabela (código x cultura) da classificação C81 da tabela 1612
if "soup" not in globals() or soup is None or "lstClassificacoes_lstCategorias_0_lblIdCategoria_0" not in str(soup):
    url = "https://apisidra.ibge.gov.br/desctabapi.aspx?c=1612"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

ids = soup.select('span[id^="lstClassificacoes_lstCategorias_0_lblIdCategoria_"]')
nomes = soup.select('span[id^="lstClassificacoes_lstCategorias_0_lblNomeCategoria_"]')

n = min(len(ids), len(nomes))
tabela_categorias_1612 = pd.DataFrame(
    {
        "codigo": [ids[i].get_text(strip=True) for i in range(n)],
        "cultura": [nomes[i].get_text(strip=True) for i in range(n)],
    }
)

tabela_categorias_1612["codigo"] = pd.to_numeric(tabela_categorias_1612["codigo"], errors="coerce")
tabela_categorias_1612 = tabela_categorias_1612.sort_values("codigo", na_position="last").reset_index(drop=True)

tabela_categorias_1612

In [50]:
from bs4 import BeautifulSoup
import re

def obter_tabela_categorias_sidra(
    codigo_tabela: int | str,
    codigo_classificacao: int | str | None = None,
    headers: dict | None = None,
    timeout: int = 30,
) -> pd.DataFrame:
    """
    Extrai tabela (codigo x categoria) de uma classificação da desctabapi do SIDRA.
    Se `codigo_classificacao` for None, usa a primeira classificação da tabela.
    """
    url = f"https://apisidra.ibge.gov.br/desctabapi.aspx?c={codigo_tabela}"
    resp = requests.get(url, headers=headers or HEADERS, timeout=timeout)
    resp.raise_for_status()
    soup_local = BeautifulSoup(resp.text, "html.parser")

    # Descobre índice da classificação na página (0, 1, 2, ...)
    spans_class = soup_local.select('span[id^="lstClassificacoes_lblIdClassificacao_"]')
    idx_por_codigo = {}
    for sp in spans_class:
        m = re.search(r"_(\d+)$", sp.get("id", ""))
        if m:
            idx_por_codigo[str(sp.get_text(strip=True))] = int(m.group(1))

    if codigo_classificacao is None:
        idx = 0
    else:
        idx = idx_por_codigo.get(str(codigo_classificacao))
        if idx is None:
            disponiveis = sorted(idx_por_codigo.keys(), key=lambda x: int(x))
            raise ValueError(
                f"Classificação {codigo_classificacao} não encontrada na tabela {codigo_tabela}. "
                f"Disponíveis: {disponiveis}"
            )

    ids = soup_local.select(f'span[id^="lstClassificacoes_lstCategorias_{idx}_lblIdCategoria_"]')
    nomes = soup_local.select(f'span[id^="lstClassificacoes_lstCategorias_{idx}_lblNomeCategoria_"]')

    n = min(len(ids), len(nomes))
    tabela = pd.DataFrame(
        {
            "codigo": [ids[i].get_text(strip=True) for i in range(n)],
            "cultura": [nomes[i].get_text(strip=True) for i in range(n)],
        }
    )

    tabela["codigo"] = pd.to_numeric(tabela["codigo"], errors="coerce").astype("Int64")
    return tabela.sort_values("codigo", na_position="last").reset_index(drop=True)



In [51]:
# Exemplo: tabela 1612, classificação C81
tabela_codigos_cultura_1612 = obter_tabela_categorias_sidra(1612, 81)

tabela_codigos_cultura_1612

,codigo,cultura
0,0,Total
1,2688,Abacaxi*
2,2689,Algodão herbáceo (em caroço)
3,2690,Alho
4,2691,Amendoim (em casca)
5,2692,Arroz (em casca)
6,2693,Aveia (em grão)
7,2694,Batata-doce
8,2695,Batata-inglesa
9,2696,Cana-de-açúcar


In [52]:
tabela_codigos_cultura_1613 = obter_tabela_categorias_sidra(1613, 82)

In [18]:
tabela_codigos_cultura_1613

,codigo,cultura
0,0,Total
1,2717,Abacate
2,2718,Algodão arbóreo (em caroço)
3,2719,Azeitona
4,2720,Banana (cacho)
5,2721,Borracha (látex coagulado)
6,2722,Cacau (em amêndoa)
7,2723,Café (em grão) Total
8,2724,Caqui
9,2725,Castanha de caju


Tirar as culturas da tabela 1612 com nome de Algodão, Amendoim, Arroz, Aveia, Cana-de-açúcar, Centeio, Cevada, Feijão, Girassol, Mamona, Milho, Soja, Sorgo, Trigo, Triticale. 

Tirar as culturas da tabela 1613 com nome de Cana para forragem, Alfafa fenada, Café Arábica e Canephora, Borracha (látex líquido), Caju. 

In [56]:
# Filtrar culturas da tabela 1612
culturas_1612_filtro = [
    "Algodão herbáceo (em caroço)",
    "Amendoim (em casca)",
    "Arroz (em casca)",
    "Aveia (em grão)",
    "Cana-de-açúcar",
    "Centeio (em grão)",
    "Cevada (em grão)",
    "Feijão (em grão)",
    "Girassol (em grão)",
    "Mamona (baga)",
    "Milho (em grão)",
    "Soja (em grão)",
    "Sorgo (em grão)",
    "Trigo (em grão)",
    "Triticale (em grão)",
    "Alfafa fenada",
    "Cana para forragem",
]

def excluir_culturas(df: pd.DataFrame, culturas: list[str], coluna: str = "cultura") -> pd.DataFrame:
    padrao = "|".join(re.escape(c) for c in culturas)
    return df[
        ~df[coluna].str.contains(padrao, case=False, regex=True, na=False)
    ].reset_index(drop=True)

# def filtrar_culturas(df: pd.DataFrame, culturas: list[str], coluna: str = "cultura") -> pd.DataFrame:
#     padrao = "|".join(re.escape(c) for c in culturas)
#     return df[
#         df[coluna].str.contains(padrao, case=False, regex=True, na=False)
#     ].reset_index(drop=True)

tabela_1612_filtrada = excluir_culturas(
    tabela_codigos_cultura_1612,
    culturas_1612_filtro
)

# Filtrar culturas da tabela 1613
culturas_1613_filtro = [ 
    "Canephora",
    "Arábica",  
    "Borracha (látex líquido)",
    "Caju"
]

tabela_1613_filtrada = excluir_culturas(
    obter_tabela_categorias_sidra(1613, 82),
    culturas_1613_filtro
)

print(f"Tabela 1612 filtrada: {len(tabela_1612_filtrada)} culturas")
print(f"Tabela 1613 filtrada: {len(tabela_1613_filtrada)} culturas")

# display(tabela_1612_filtrada)
# display(tabela_1613_filtrada)

Tabela 1612 filtrada: 17 culturas
Tabela 1613 filtrada: 34 culturas


In [58]:
salvar_dataframe_excel(tabela_1612_filtrada, "cultura_filtrada_1612.xlsx")
# salvar_dataframe_excel(tabela_1613_filtrada, "cultura_filtrada_1613.xlsx")


Excel salvo em: /home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/cultura_filtrada_1612.xlsx (engine=openpyxl)


PosixPath('/home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/cultura_filtrada_1612.xlsx')

Baixando todos os dados referente as lavouras de pela CONAB, para comprar os resultados com os dados do IBGE.

In [28]:
# Coleta e padronização das séries históricas de área da CONAB (aba "Área")
import tempfile
import os
import re
import xlrd

fontes_conab = {
    "cana_de_acucar": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/cana-de-acucar/area-total/canaseriehist-area-total.xls",
    "cafe_total": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/cafe/total-arabica-e-conilon/cafetotalseriehist.xls",
    "algodao": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/algodao/algodaoseriehist.xls/@@download/file",
    "amendoim_1a_safra": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/amendoim/amendoim1aseriehist.xls",
    "amendoim_2a_safra": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/amendoim/amendoim2aseriehist.xls",
    "arroz": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/arroz/arroztotalseriehist.xls",
    "aveia": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/aveia/aveiaseriehist.xls",
    "centeio": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/centeio/centeioseriehist.xls/",
    "cevada": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/cevada/cevadaseriehist.xls",
    "feijao_1a_safra": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/feijao/feijao1aseriehist.xls",
    "feijao_2a_safra": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/feijao/feijao2aseriehist.xls", 
    "feijao_3a_safra": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/feijao/feijao3aseriehist.xls",
    "gergelim": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/gergelim/gergelimseriehist.xls",
    "girassol": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/girassol/girassolseriehist.xls",
    "mamona": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/mamona/mamonaseriehist.xls",
    "milho_1a_safra": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/milho/milho1aseriehist.xls",
    "milho_2a_safra": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/milho/milho2aseriehist.xls",
    "milho_3a_safra": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/milho/milho3aseriehist.xls",
    "soja": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/soja/sojaseriehist.xls",
    "sorgo": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/sorgo/sorgoseriehist.xls",
    "trigo": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/trigo/trigoseriehist.xls",
    "triticale": "https://www.gov.br/conab/pt-br/atuacao/informacoes-agropecuarias/safras/series-historicas/graos/triticale/triticaleseriehist.xls"
}

def _safra_para_ano(col):
    s = str(col).strip()
    # Formato YYYY ou YYYY.0 (ex: 2001 ou 2001.0 - café CONAB)
    m_y = re.fullmatch(r"(\d{4})(\.0)?", s)
    if m_y:
        return int(m_y.group(1))
    # Formato YYYY/YYYY (ex: 2001/2002)
    m4 = re.search(r"(\d{4})\s*/\s*(\d{4})$", s)
    if m4:
        return int(m4.group(2))
    # Formato YYYY/YY (ex: 2001/02)
    m = re.search(r"(\d{4})\s*/\s*(\d{2})$", s)
    if not m:
        return None
    ano_ini = int(m.group(1))
    aa = int(m.group(2))
    seculo = (ano_ini // 100) * 100
    ano_fim = seculo + aa
    if ano_fim < ano_ini:
        ano_fim += 100
    return ano_fim

def _baixar_bytes(url: str) -> tuple[bytes, str]:
    """Baixa a URL e retorna (conteúdo, extensão)."""
    url_limpa = url.replace("/@@download/file", "")
    resp = requests.get(url_limpa, headers=HEADERS, timeout=60, allow_redirects=True)
    resp.raise_for_status()
    content_type = resp.headers.get("Content-Type", "")
    ext = ".xlsx" if ("spreadsheetml" in content_type or url_limpa.lower().endswith(".xlsx")) else ".xls"
    return resp.content, ext

def _xls_para_dataframe(conteudo: bytes, aba: int) -> pd.DataFrame:
    """Lê .xls OLE2 diretamente com xlrd usando sempre a primeira aba."""
    book = xlrd.open_workbook(file_contents=conteudo)
    if book.nsheets == 0:
        raise ValueError("Arquivo .xls sem abas.")

    sheet = book.sheet_by_index(aba)

    linhas = []
    for i in range(sheet.nrows):
        row = []
        for j in range(sheet.ncols):
            cell = sheet.cell(i, j)
            if cell.ctype == xlrd.XL_CELL_DATE:
                row.append(xlrd.xldate_as_datetime(cell.value, book.datemode))
            else:
                row.append(cell.value)
        linhas.append(row)

    df = pd.DataFrame(linhas)
    df = df.iloc[5:].reset_index(drop=True)
    df = df.iloc[:-2].reset_index(drop=True)
    return df

def _ler_area_conab(url: str, cultura: str) -> pd.DataFrame:
    conteudo, ext = _baixar_bytes(url)

    if ext == ".xlsx":
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".xlsx")
        tmp.write(conteudo)
        tmp.close()
        try:
            bruto = pd.read_excel(tmp.name, sheet_name=0, header=5, footer=2, engine="openpyxl")
        finally:
            os.unlink(tmp.name)
    else:
        # .xls OLE2: usa xlrd diretamente para contornar restrição do pandas 2.x
        bruto = _xls_para_dataframe(conteudo, aba=0)

    # Localiza a linha de cabeçalho:
    # - se a 1ª linha já tiver anos no formato YYYY ou YYYY.0, usa-a diretamente
    # - caso contrário, procura linha com safras no formato YYYY/YY ou YYYY/YYYY
    cab_idx = None

    if not bruto.empty:
        primeira_linha = bruto.iloc[0].astype(str).str.strip()
        if primeira_linha.str.fullmatch(r"\d{4}(\.0)?").any():
            cab_idx = 0
        else:
            for i, row in bruto.iterrows():
                row_str = row.astype(str).str.strip()
                if row_str.str.contains(r"\d{4}/\d{2,4}", regex=True).any():
                    cab_idx = i
                    break
    if cab_idx is None:
        raise ValueError(f"Linha de cabeçalho com safras não encontrada em: {url}")

    bruto.columns = bruto.iloc[cab_idx].astype(str).str.strip()
    tab = bruto.iloc[cab_idx + 1:].copy().reset_index(drop=True)
    tab = tab.dropna(axis=1, how="all").dropna(axis=0, how="all")

    primeira_col = tab.columns[0]
    tab = tab.rename(columns={primeira_col: "REGIÃO/UF"})

    # Normalizar nomes de colunas: "2001.0" → "2001"
    tab.columns = [re.sub(r"^(\d{4})\.0$", r"\1", str(c).strip()) for c in tab.columns]

    colunas_safra = [c for c in tab.columns if _safra_para_ano(c) is not None]
    base = tab[["REGIÃO/UF"] + colunas_safra].copy()
    base = base.melt(
        id_vars="REGIÃO/UF",
        value_vars=colunas_safra,
        var_name="safra",
        value_name="valor",
    )

    base["ano"] = base["safra"].apply(_safra_para_ano)
    base["valor"] = pd.to_numeric(
        base["valor"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce",
    )

    base["cultura"] = cultura
    base["unidade"] = "1000 Hectares"
    base["fonte"] = "CONAB"

    return base[["fonte", "cultura", "REGIÃO/UF", "safra", "ano", "unidade", "valor"]]

# Coleta consolidada
blocos = []
for cultura, link in fontes_conab.items():
    try:
        df_tmp = _ler_area_conab(link, cultura)
        blocos.append(df_tmp)
        print(f"✓ {cultura}: {len(df_tmp)} linhas")
    except Exception as e:
        print(f"✗ {cultura}: {e}")

df_conab_areas = pd.concat(blocos, ignore_index=True) if blocos else pd.DataFrame()
df_conab_areas


✓ cana_de_acucar: 700 linhas
✓ cafe_total: 675 linhas
✓ algodao: 1715 linhas
✓ amendoim_1a_safra: 1715 linhas
✓ amendoim_2a_safra: 1715 linhas
✓ arroz: 1715 linhas
✓ aveia: 1750 linhas
✓ centeio: 1750 linhas
✓ cevada: 1750 linhas
✓ feijao_1a_safra: 1715 linhas
✓ feijao_2a_safra: 1715 linhas
✓ feijao_3a_safra: 1715 linhas
✓ gergelim: 1715 linhas
✓ girassol: 1715 linhas
✓ mamona: 1715 linhas
✓ milho_1a_safra: 1715 linhas
✓ milho_2a_safra: 1715 linhas
✓ milho_3a_safra: 1715 linhas
✓ soja: 1715 linhas
✓ sorgo: 1715 linhas
✓ trigo: 1750 linhas
✓ triticale: 1750 linhas


,fonte,cultura,REGIÃO/UF,safra,ano,unidade,valor
0,CONAB,cana_de_acucar,NORTE,2005/06,2006,1000 Hectares,186.0
1,CONAB,cana_de_acucar,RR,2005/06,2006,1000 Hectares,0.0
2,CONAB,cana_de_acucar,RO,2005/06,2006,1000 Hectares,0.0
3,CONAB,cana_de_acucar,AC,2005/06,2006,1000 Hectares,0.0
4,CONAB,cana_de_acucar,AM,2005/06,2006,1000 Hectares,38.0
...,...,...,...,...,...,...,...
35845,CONAB,triticale,SC,2025,2025,1000 Hectares,0.0
35846,CONAB,triticale,RS,2025,2025,1000 Hectares,36.0
35847,CONAB,triticale,NORTE/NORDESTE,2025,2025,1000 Hectares,0.0
35848,CONAB,triticale,CENTRO-SUL,2025,2025,1000 Hectares,114.0


In [66]:
# Transformar df_conab_areas para formato similar a tabela_pivot_1612
df_conab_pivot = df_conab_areas.copy()

# Renomear colunas para ficar compatível
df_conab_pivot = df_conab_pivot.rename(columns={
    "REGIÃO/UF": "sigla_uf",
    "cultura": "cultura",
})

# Adicionar colunas ausentes para compatibilidade
_geocodigo_map = {
    "AC": 12, "AL": 27, "AP": 16, "AM": 13, "BA": 29, "CE": 23,
    "DF": 53, "ES": 32, "GO": 52, "MA": 21, "MT": 51, "MS": 50,
    "MG": 31, "PA": 15, "PB": 25, "PR": 41, "PE": 26, "PI": 22,
    "RJ": 33, "RN": 24, "RS": 43, "RO": 11, "RR": 14, "SC": 42,
    "SP": 35, "SE": 28, "TO": 17,
}

df_conab_pivot["geocodigo"] = df_conab_pivot["sigla_uf"].map(_geocodigo_map)
df_conab_pivot["nome_uf"] = df_conab_pivot["sigla_uf"].map({
    "AC": "Acre", "AL": "Alagoas", "AP": "Amapá", "AM": "Amazonas", "BA": "Bahia", "CE": "Ceará",
    "DF": "Distrito Federal", "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão", "MT": "Mato Grosso", "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais", "PA": "Pará", "PB": "Paraíba", "PR": "Paraná", "PE": "Pernambuco", "PI": "Piauí",
    "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte", "RS": "Rio Grande do Sul", "RO": "Rondônia", "RR": "Roraima", "SC": "Santa Catarina",
    "SP": "São Paulo", "SE": "Sergipe", "TO": "Tocantins",
})
df_conab_pivot["producao"] = "Área colhida"

colunas_chave = ["geocodigo", "nome_uf", "cultura", "producao", "unidade", "fonte"]

tabela_pivot_conab = df_conab_pivot.pivot_table(
    index=colunas_chave,
    columns="ano",
    values="valor",
    aggfunc="sum",
    fill_value=0,
).reset_index()



tabela_pivot_conab.columns.name = None

colunas_ano = sorted([c for c in tabela_pivot_conab.columns if c not in colunas_chave], key=int)
tabela_pivot_conab = tabela_pivot_conab[colunas_chave + colunas_ano]

tabela_pivot_conab

,geocodigo,nome_uf,cultura,producao,unidade,fonte,1976,1977,1978,1979,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,11.0,Rondônia,algodao,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,52.0,98.0,81.0,81.0,91.0,87.0,88.0
1,11.0,Rondônia,amendoim_1a_safra,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,11.0,Rondônia,amendoim_2a_safra,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,11.0,Rondônia,arroz,Área colhida,1000 Hectares,CONAB,0.0,737.0,652.0,705.0,...,426.0,406.0,424.0,424.0,425.0,368.0,329.0,373.0,412.0,440.0
4,11.0,Rondônia,aveia,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
573,53.0,Distrito Federal,milho_3a_safra,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
574,53.0,Distrito Federal,soja,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,700.0,700.0,715.0,732.0,745.0,785.0,842.0,861.0,887.0,884.0
575,53.0,Distrito Federal,sorgo,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,33.0,70.0,72.0,57.0,99.0,98.0,90.0,180.0,180.0,185.0
576,53.0,Distrito Federal,trigo,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,8.0,9.0,23.0,24.0,26.0,28.0,32.0,34.0,70.0,74.0


In [67]:
# Remove linhas com cultura contendo etapa de safra (1a_safra, 2a_safra, 3a_safra)
padrao_safra = r"_(1a|2a|3a)_safra$"

if "tabela_conab_pivot" in globals():
    tabela_pivot_conab = tabela_pivot_conab[
        ~tabela_pivot_conab["cultura"].str.contains(padrao_safra, case=False, regex=True, na=False)
    ].reset_index(drop=True)
    display(tabela_pivot_conab.head())
elif "tabela_pivot_conab" in globals():
    tabela_pivot_conab = tabela_pivot_conab[
        ~tabela_pivot_conab["cultura"].str.contains(padrao_safra, case=False, regex=True, na=False)
    ].reset_index(drop=True)
    display(tabela_pivot_conab.head())
else:
    print("Nenhuma tabela encontrada: 'tabela_conab_pivot' ou 'tabela_pivot_conab'.")

/tmp/ipykernel_199255/1555269147.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~tabela_pivot_conab["cultura"].str.contains(padrao_safra, case=False, regex=True, na=False)


,geocodigo,nome_uf,cultura,producao,unidade,fonte,1976,1977,1978,1979,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,11.0,Rondônia,algodao,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,52.0,98.0,81.0,81.0,91.0,87.0,88.0
1,11.0,Rondônia,arroz,Área colhida,1000 Hectares,CONAB,0.0,737.0,652.0,705.0,...,426.0,406.0,424.0,424.0,425.0,368.0,329.0,373.0,412.0,440.0
2,11.0,Rondônia,aveia,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,11.0,Rondônia,cafe_total,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,876570.0,742550.0,638790.0,627290.0,635690.0,635690.0,649770.0,606210.0,398050.0,407620.0
4,11.0,Rondônia,cana_de_acucar,Área colhida,1000 Hectares,CONAB,0.0,0.0,0.0,0.0,...,434.0,342.0,182.0,127.0,0.0,0.0,0.0,0.0,0.0,0.0


In [46]:
salvar_dataframe_excel(tabela_pivot_conab, "tabela_conab_areas.xlsx")

Excel salvo em: /home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/tabela_conab_areas.xlsx (engine=openpyxl)


PosixPath('/home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/tabela_conab_areas.xlsx')

In [36]:
# Reformatar df_conab_areas para formato similar a t1618:
# linhas por (UF, cultura, ano) e colunas 1ª Safra / 2ª Safra / 3ª Safra
df_base = df_conab_areas.copy()

# Manter apenas culturas com etapa explícita no nome
df_base = df_base[df_base["cultura"].str.contains(r"_(1a|2a|3a)_safra$", case=False, regex=True, na=False)].copy()

# Extrair etapa (1a/2a/3a) e mapear para rótulo final
df_base["etapa"] = df_base["cultura"].str.extract(r'_(1a|2a|3a)_safra$', flags=re.IGNORECASE, expand=False).str.lower()
df_base["etapa"] = df_base["etapa"].map({"1a": "1ª Safra", "2a": "2ª Safra", "3a": "3ª Safra"})

# Limpar nome da cultura (remove sufixo de etapa)
df_base["cultura"] = (
    df_base["cultura"]
    .str.replace(r'_(1a|2a|3a)_safra$', '', regex=True)
    .str.replace("_", " ")
    .str.strip()
)

df_base = df_base.rename(columns={"REGIÃO/UF": "nome_uf"})
df_base["geocodigo"] = df_base["nome_uf"].map(_geocodigo_map)
df_base["producao"] = "Área colhida"

# Ano fica no índice; etapas viram colunas de safra
index_cols = ["geocodigo", "nome_uf", "producao", "cultura", "ano", "unidade"]

df_conab_1618_like = (
    df_base.groupby(index_cols + ["etapa"], observed=True)["valor"]
    .sum()
    .unstack("etapa", fill_value=0)
    .reset_index()
)

df_conab_1618_like.columns.name = None
for c in ["1ª Safra", "2ª Safra", "3ª Safra"]:
    if c not in df_conab_1618_like.columns:
        df_conab_1618_like[c] = 0.0

df_conab_1618_like = df_conab_1618_like[index_cols + ["1ª Safra", "2ª Safra", "3ª Safra"]]
df_conab_1618_like = df_conab_1618_like.sort_values(["nome_uf", "cultura", "ano"]).reset_index(drop=True)

df_conab_1618_like


/tmp/ipykernel_199255/1971983615.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_base = df_base[df_base["cultura"].str.contains(r"_(1a|2a|3a)_safra$", case=False, regex=True, na=False)].copy()


,geocodigo,nome_uf,producao,cultura,ano,unidade,1ª Safra,2ª Safra,3ª Safra
0,12.0,AC,Área colhida,amendoim,1977,1000 Hectares,0.0,0.0,0.0
1,12.0,AC,Área colhida,amendoim,1978,1000 Hectares,0.0,0.0,0.0
2,12.0,AC,Área colhida,amendoim,1979,1000 Hectares,0.0,0.0,0.0
3,12.0,AC,Área colhida,amendoim,1980,1000 Hectares,0.0,0.0,0.0
4,12.0,AC,Área colhida,amendoim,1981,1000 Hectares,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
3964,17.0,TO,Área colhida,milho,2021,1000 Hectares,409.0,2253.0,0.0
3965,17.0,TO,Área colhida,milho,2022,1000 Hectares,454.0,3247.0,0.0
3966,17.0,TO,Área colhida,milho,2023,1000 Hectares,574.0,3734.0,0.0
3967,17.0,TO,Área colhida,milho,2024,1000 Hectares,603.0,3349.0,0.0


In [68]:
salvar_dataframe_excel(df_conab_1618_like, "conab_safras_areas.xlsx")

Excel salvo em: /home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/conab_safras_areas.xlsx (engine=openpyxl)


PosixPath('/home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/conab_safras_areas.xlsx')

In [64]:
# juntar todos os excel em um com cada excel como uma aba diferente
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
def salvar_multiplas_tabelas_excel(tabelas: dict[str, pd.DataFrame], nome_arquivo="tabelas_consolidadas.xlsx", pasta=None):
    pasta = Path(pasta) if pasta else Path.cwd()
    pasta.mkdir(parents=True, exist_ok=True)
    caminho = pasta / nome_arquivo

    wb = Workbook()
    for nome, df in tabelas.items():
        ws = wb.create_sheet(title=nome[:31])  # Limite de 31 caracteres para nome da aba
        for r in dataframe_to_rows(df, index=False, header=True):
            ws.append(r)

    # Remove a aba padrão criada pelo Workbook
    if "Sheet" in wb.sheetnames:
        del wb["Sheet"]

    wb.save(caminho)
    print(f"Excel consolidado salvo em: {caminho.resolve()}")
    return caminho

In [71]:
salvar_multiplas_tabelas_excel(tabelas={"t1618": t1618,
                                        "t1612_ano": tabela_pivot_1612,
                                        "t1613_ano": tabela_pivot_1613,
                                        "t1612_culturas": tabela_codigos_cultura_1612,
                                        "t1613_culturas": tabela_codigos_cultura_1613,
                                        "t1612_culturas_filtadas": tabela_1612_filtrada,
                                        "t1613_culturas_filtradas": tabela_1613_filtrada,
                                        "tabela_1618": t1618,
                                        "conab_cana_cafe_graos": tabela_pivot_conab,
                                        "conab_safras": df_conab_1618_like
                                        },
                               nome_arquivo="tabelas_consolidadas.xlsx")

Excel consolidado salvo em: /home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/tabelas_consolidadas.xlsx


PosixPath('/home/daianasales/vscode/ISAGRO/IS_Agro_day/scripts/tabelas_consolidadas.xlsx')